# No Scaling, No PCA Classification
## Random Forest and XGBoost Models
Training and evaluating models on non-scaled dataset with train/validation/test split.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve, auc
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import joblib
import os

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Explore Dataset

In [ ]:
# Load dataset (no scaling, no PCA)
df = pd.read_csv('../datasets/trojan_cleaned(no_scaling).csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 2. Data Preparation

In [ ]:
# Separate features and target
# Assuming the last column is the target (adjust if needed)
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

## 3. Train/Validation/Test Split

In [ ]:
# Splitting Dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, 
    y, 
    test_size=0.30,
    random_state=42
)

# Splitting the 30% --> 2 x 15% (test and val)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50,
    random_state=42
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Validation set size: {X_val.shape[0]} ({X_val.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print(f"\nTotal: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} samples")

## 4. Random Forest Model

In [ ]:
# Train Random Forest
print("="*50)
print("RANDOM FOREST")
print("="*50)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

# Probabilities for ROC curve
rf_train_proba = rf_model.predict_proba(X_train)[:, 1]
rf_val_proba = rf_model.predict_proba(X_val)[:, 1]
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("\nTRAINING SET:")
print(f"Accuracy: {accuracy_score(y_train, rf_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, rf_train_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_train, rf_train_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_train, rf_train_pred, zero_division=0):.4f}")

print("\nVALIDATION SET:")
rf_val_acc = accuracy_score(y_val, rf_val_pred)
print(f"Accuracy: {rf_val_acc:.4f}")
print(f"Precision: {precision_score(y_val, rf_val_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, rf_val_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_val, rf_val_pred, zero_division=0):.4f}")

print("\nTEST SET:")
rf_test_acc = accuracy_score(y_test, rf_test_pred)
print(f"Accuracy: {rf_test_acc:.4f}")
print(f"Precision: {precision_score(y_test, rf_test_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, rf_test_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, rf_test_pred, zero_division=0):.4f}")

# Confusion Matrix
rf_cm = confusion_matrix(y_test, rf_test_pred)
print("\nConfusion Matrix:")
print(rf_cm)

## 5. XGBoost Model

In [ ]:
# Train XGBoost
print("\n" + "="*50)
print("XGBOOST")
print("="*50)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# Predictions
xgb_train_pred = xgb_model.predict(X_train)
xgb_val_pred = xgb_model.predict(X_val)
xgb_test_pred = xgb_model.predict(X_test)

# Probabilities for ROC curve
xgb_train_proba = xgb_model.predict_proba(X_train)[:, 1]
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("\nTRAINING SET:")
print(f"Accuracy: {accuracy_score(y_train, xgb_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, xgb_train_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_train, xgb_train_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_train, xgb_train_pred, zero_division=0):.4f}")

print("\nVALIDATION SET:")
xgb_val_acc = accuracy_score(y_val, xgb_val_pred)
print(f"Accuracy: {xgb_val_acc:.4f}")
print(f"Precision: {precision_score(y_val, xgb_val_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, xgb_val_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_val, xgb_val_pred, zero_division=0):.4f}")

print("\nTEST SET:")
xgb_test_acc = accuracy_score(y_test, xgb_test_pred)
print(f"Accuracy: {xgb_test_acc:.4f}")
print(f"Precision: {precision_score(y_test, xgb_test_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, xgb_test_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, xgb_test_pred, zero_division=0):.4f}")

# Confusion Matrix
xgb_cm = confusion_matrix(y_test, xgb_test_pred)
print("\nConfusion Matrix:")
print(xgb_cm)

## 6. Model Comparison and Visualization

In [ ]:
# Comparison metrics
models = ['Random Forest', 'XGBoost']
test_accuracies = [rf_test_acc, xgb_test_acc]
test_precisions = [
    precision_score(y_test, rf_test_pred, zero_division=0),
    precision_score(y_test, xgb_test_pred, zero_division=0)
]
test_recalls = [
    recall_score(y_test, rf_test_pred, zero_division=0),
    recall_score(y_test, xgb_test_pred, zero_division=0)
]
test_f1s = [
    f1_score(y_test, rf_test_pred, zero_division=0),
    f1_score(y_test, xgb_test_pred, zero_division=0)
]

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy comparison
axes[0, 0].bar(models, test_accuracies, color=['#1f77b4', '#ff7f0e'])
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(test_accuracies):
    axes[0, 0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Precision comparison
axes[0, 1].bar(models, test_precisions, color=['#1f77b4', '#ff7f0e'])
axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim([0, 1])
for i, v in enumerate(test_precisions):
    axes[0, 1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Recall comparison
axes[1, 0].bar(models, test_recalls, color=['#1f77b4', '#ff7f0e'])
axes[1, 0].set_ylabel('Recall', fontsize=12)
axes[1, 0].set_title('Recall Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylim([0, 1])
for i, v in enumerate(test_recalls):
    axes[1, 0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# F1-Score comparison
axes[1, 1].bar(models, test_f1s, color=['#1f77b4', '#ff7f0e'])
axes[1, 1].set_ylabel('F1-Score', fontsize=12)
axes[1, 1].set_title('F1-Score Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim([0, 1])
for i, v in enumerate(test_f1s):
    axes[1, 1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Random Forest
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Random Forest\nConfusion Matrix', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# XGBoost
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=False)
axes[1].set_title('XGBoost\nConfusion Matrix', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Random Forest ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_test_proba)
roc_auc_rf = auc(fpr_rf, tpr_rf)
axes[0].plot(fpr_rf, tpr_rf, color='#1f77b4', lw=2, label=f'AUC = {roc_auc_rf:.4f}')
axes[0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Random Forest\nROC Curve', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_test_proba)
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)
axes[1].plot(fpr_xgb, tpr_xgb, color='#ff7f0e', lw=2, label=f'AUC = {roc_auc_xgb:.4f}')
axes[1].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('XGBoost\nROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAUC-ROC Scores (Test Set):")
print(f"Random Forest: {roc_auc_rf:.4f}")
print(f"XGBoost: {roc_auc_xgb:.4f}")

## 7. Feature Importance

In [ ]:
# Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest Feature Importance
rf_importance = rf_model.feature_importances_
rf_indices = np.argsort(rf_importance)[-20:]  # Top 20 features
axes[0].barh(range(len(rf_indices)), rf_importance[rf_indices], color='#1f77b4')
axes[0].set_yticks(range(len(rf_indices)))
axes[0].set_yticklabels([f'Feature {i}' for i in rf_indices])
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('Random Forest - Top 20 Feature Importance', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

# XGBoost Feature Importance
xgb_importance = xgb_model.feature_importances_
xgb_indices = np.argsort(xgb_importance)[-20:]  # Top 20 features
axes[1].barh(range(len(xgb_indices)), xgb_importance[xgb_indices], color='#ff7f0e')
axes[1].set_yticks(range(len(xgb_indices)))
axes[1].set_yticklabels([f'Feature {i}' for i in xgb_indices])
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('XGBoost - Top 20 Feature Importance', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 8. Export Trained Models

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('models/no_scaling_no_pca', exist_ok=True)

# Save models using joblib
model_path_rf = 'models/no_scaling_no_pca/random_forest_model.pkl'
model_path_xgb = 'models/no_scaling_no_pca/xgboost_model.pkl'

joblib.dump(rf_model, model_path_rf)
joblib.dump(xgb_model, model_path_xgb)

print("Models saved successfully!")
print(f"✓ Random Forest: {model_path_rf}")
print(f"✓ XGBoost: {model_path_xgb}")

In [ ]:
# Summary Report
print("\n" + "="*60)
print("SUMMARY REPORT - NO SCALING, NO PCA")
print("="*60)
print(f"\nBest Model (by Test Accuracy): ", end="")
best_model_idx = np.argmax(test_accuracies)
print(f"{models[best_model_idx]} ({test_accuracies[best_model_idx]:.4f})")

print("\nTest Set Performance:")
print(f"{'Model':<20} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 55)
for i, model in enumerate(models):
    print(f"{model:<20} {test_accuracies[i]:<12.4f} {test_precisions[i]:<12.4f} {test_recalls[i]:<12.4f} {test_f1s[i]:<12.4f}")

print(f"\n{'Model':<20} {'AUC-ROC':<12}")
print("-" * 35)
print(f"{'Random Forest':<20} {roc_auc_rf:<12.4f}")
print(f"{'XGBoost':<20} {roc_auc_xgb:<12.4f}")